## Sequentail chains

In [53]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [54]:
load_dotenv()

True

In [55]:
api_key = os.getenv('GOOGLE_API_KEY')

In [56]:
import google.generativeai as genai
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))


In [57]:
models = genai.list_models()


In [58]:

for m in models:
    print(m.name, "->", m.supported_generation_methods)

models/gemini-2.5-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-pro -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-exp-image-generation -> ['generateContent', 'countTokens', 'bidiGenerateContent']
models/gemini-2.0-flash-lite-001 -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-lite -> ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.5-flash-preview-tts -> ['countTokens', 'generateContent']
models/gemini-2.5-pro-preview-tts -> ['countTokens', 'generateContent', 'batchGenerateContent']
models/gemma-3-1b-it -> ['generateContent', 'countToke

In [59]:
model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key = api_key
)

In [60]:
# create the state 

class LLMState(TypedDict):
    question:str
    answer : str

In [61]:
def llm_qa(state:LLMState) -> LLMState:
    # extra a question from the state
    question = state['question']

    # from a prompt
    prompt = f"answer the following question {question}"

    # Ask the question to the llm
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer
    return state

In [62]:
# create out graph
graph = StateGraph(LLMState)

# add the nodes
graph.add_node('llm_qa',llm_qa)

# add the edge
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# Comilpe the graph
workflow = graph.compile()



In [63]:
# execute

intial_state = {'question':'how moon is far from the earth'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'question': 'how moon is far from the earth', 'answer': "The average distance from the Earth to the Moon is approximately **384,400 kilometers (238,900 miles)**.\n\nHowever, this distance is not constant because the Moon's orbit around the Earth is elliptical (oval-shaped), not a perfect circle.\n\n*   **At its closest point (perigee):** The Moon can be as close as about 363,104 km (225,623 miles).\n*   **At its farthest point (apogee):** The Moon can be as far as about 405,696 km (252,088 miles).\n\nSo, while it varies, the number most often cited and a good general reference is the average of roughly **384,400 km (238,900 miles)**."}


In [64]:
final_state['answer']

"The average distance from the Earth to the Moon is approximately **384,400 kilometers (238,900 miles)**.\n\nHowever, this distance is not constant because the Moon's orbit around the Earth is elliptical (oval-shaped), not a perfect circle.\n\n*   **At its closest point (perigee):** The Moon can be as close as about 363,104 km (225,623 miles).\n*   **At its farthest point (apogee):** The Moon can be as far as about 405,696 km (252,088 miles).\n\nSo, while it varies, the number most often cited and a good general reference is the average of roughly **384,400 km (238,900 miles)**."